# 02 - HITL Active Learning Loop

Re-run this notebook after each human labelling round.
Increment `PENDING_BATCH_TO_PROCESS` each time (1 → 2 → 3 → 4).

In [ ]:
%%time
import os
from pathlib import Path
import sys

# --- ENVIRONMENT SWITCH ---
# True  → local machine with Google Drive Desktop mounted
# False → Google Colab cloud
RUNNING_LOCALLY = False

# --- DATASET TYPE ---
# 'AI'  → AItrust_twits_pruned_dict.json    → Partitioned Data/AI Data/
# 'Art' → AItrust_Art_pruned_twit_dict.json → Partitioned Data/Art Data/
DATASET_TYPE = 'AI'

if RUNNING_LOCALLY:
    _REPO_ROOT = str(Path(os.getcwd()).resolve().parents[1])
    if _REPO_ROOT not in sys.path:
        sys.path.insert(0, _REPO_ROOT)
    BASE_PATH = Path('/Volumes/GoogleDrive/My Drive/Colab Projects/AI Public Trust')
else:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/My Drive/Colab Projects/AI Public Trust')

# Pre-compute critical paths
twits_folder          = BASE_PATH / 'Raw Data/Twits/'
test_folder           = BASE_PATH / 'Raw Data/'
datasets_folder       = BASE_PATH / 'Data Sets'
cleanedds_folder      = BASE_PATH / 'Data Sets/Cleaned Data'
networks_folder       = BASE_PATH / 'Data Sets/Networks/'
literature_folder     = BASE_PATH / 'Literature/'
topic_models_folder   = BASE_PATH / 'Models/Topic Modeling/'
classifiers_folder    = BASE_PATH / 'Models/Classifiers/'
classifiers_folder.mkdir(parents=True, exist_ok=True)
partitioned_folder    = cleanedds_folder / 'Partitioned Data' / f'{DATASET_TYPE} Data'
hitl_folder = datasets_folder / 'Classifiers_Data' / 'HITL'

In [ ]:
%%time
if not RUNNING_LOCALLY:
    print('Running Colab setup...')
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'transformers', 'torch',
                    'sentence-transformers', 'lightgbm', 'scikit-learn', 'datasets'])
else:
    print('Running locally: skipping Colab setup.')

In [ ]:
%%time
import time
import glob
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from sklearn.multiclass import OneVsRestClassifier
import lightgbm as lgb
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          Trainer, TrainingArguments, pipeline)
from datasets import Dataset

# Configuration

In [ ]:
%%time
PENDING_BATCH_TO_PROCESS = 1   # change to 2, 3, 4 for subsequent iterations
NEXT_BATCH_PATH = partitioned_folder / f'hitl_pending_batch_{PENDING_BATCH_TO_PROCESS:02d}.pkl'

# 1. Load All Labeled Data

In [ ]:
%%time
labeled_files = sorted(glob.glob(str(hitl_folder / 'hitl_review_batch_*.csv')))
print(f'Found {len(labeled_files)} labeled batch(es).')

dfs = []
for f in labeled_files:
    tmp = pd.read_csv(f)
    human_cols = [f'human_{cat}' for cat in ['intentionalism', 'anti_intentionalism', 'cognitivism', 'expressivism', 'hedonism', 'originality', 'achievement', 'none']]
    if all(c in tmp.columns for c in human_cols):
        tmp = tmp.dropna(subset=human_cols)
        if len(tmp) > 0:
            dfs.append(tmp)

if not dfs:
    raise ValueError('No labeled data found. Label hitl_review_batch_00.csv first.')

train_df = pd.concat(dfs, ignore_index=True)
train_df['text'] = train_df['text'].astype(str)
print(f'Total labeled examples: {len(train_df):,}')

human_cols = [f'human_{cat}' for cat in ['intentionalism', 'anti_intentionalism', 'cognitivism', 'expressivism', 'hedonism', 'originality', 'achievement', 'none']]
X_tr, X_val, y_tr, y_val = train_test_split(
    train_df['text'], train_df[human_cols].values, test_size=0.1, random_state=42)

# 2. Sentence Embeddings

Generate sentence embeddings using `all-MiniLM-L6-v2` for use in the boosted tree and logistic regression.

In [ ]:
%%time
from sentence_transformers import SentenceTransformer

t0 = time.time()
st_model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings_tr  = st_model.encode(X_tr.tolist(), show_progress_bar=True)
embeddings_val = st_model.encode(X_val.tolist(), show_progress_bar=True)
print(f'Sentence embedding time: {time.time()-t0:.1f}s')

# 3. Bag-of-Words (CountVectorizer)

In [ ]:
%%time
from sklearn.feature_extraction.text import CountVectorizer

bow_vec = CountVectorizer(max_features=50_000)
X_bow_tr  = bow_vec.fit_transform(X_tr)
X_bow_val = bow_vec.transform(X_val)
print(f'BoW shape: {X_bow_tr.shape}')

# 4. Logistic Regression

In [ ]:
%%time
from sklearn.metrics import roc_auc_score

# --- On sentence embeddings ---
t0 = time.time()
lr_embed = OneVsRestClassifier(LogisticRegression(max_iter=1000))
lr_embed.fit(embeddings_tr, y_tr)
lr_embed_tr = time.time()-t0
t0 = time.time()
lr_embed_preds = lr_embed.predict(embeddings_val)
lr_embed_inf   = time.time()-t0
print(f'LR (embed) train {lr_embed_tr:.1f}s | infer {lr_embed_inf:.2f}s | F1 {f1_score(y_val, lr_embed_preds, average='macro'):.4f}')

# --- On BoW ---
t0 = time.time()
lr_bow = OneVsRestClassifier(LogisticRegression(max_iter=1000))
lr_bow.fit(X_bow_tr, y_tr)
lr_bow_tr = time.time()-t0
t0 = time.time()
lr_bow_preds = lr_bow.predict(X_bow_val)
lr_bow_inf   = time.time()-t0
print(f'LR (BoW)   train {lr_bow_tr:.1f}s | infer {lr_bow_inf:.2f}s | F1 {f1_score(y_val, lr_bow_preds, average='macro'):.4f}')

# 5. Boosted Tree (LightGBM native API)

Matches the approach in `Classifiers_Training_Final.ipynb` — uses `lgb.Dataset` and `lgb.train()` with a params dict. Run with both sentence embeddings and BoW.

In [ ]:
%%time
from lightgbm import LGBMClassifier

clf_embed = OneVsRestClassifier(LGBMClassifier(objective='binary', n_estimators=1000, verbose=-1))
t0 = time.time()
clf_embed.fit(embeddings_tr, y_tr.astype(int))
lgb_embed_tr = time.time()-t0
t0 = time.time()
lgb_embed_preds = clf_embed.predict(embeddings_val)
lgb_embed_inf   = time.time()-t0
print(f'LGB (embed) train {lgb_embed_tr:.1f}s | infer {lgb_embed_inf:.2f}s | F1 {f1_score(y_val.astype(int), lgb_embed_preds, average="macro"):.4f}')

In [ ]:
%%time
# --- On BoW ---
import scipy.sparse
X_bow_tr_dense  = X_bow_tr.toarray()  if scipy.sparse.issparse(X_bow_tr)  else X_bow_tr
X_bow_val_dense = X_bow_val.toarray() if scipy.sparse.issparse(X_bow_val) else X_bow_val

clf_bow = OneVsRestClassifier(LGBMClassifier(objective='binary', n_estimators=1000, verbose=-1))
t0 = time.time()
clf_bow.fit(X_bow_tr_dense, y_tr.astype(int))
lgb_bow_tr = time.time()-t0
t0 = time.time()
lgb_bow_preds = clf_bow.predict(X_bow_val_dense)
lgb_bow_inf   = time.time()-t0
print(f'LGB (BoW)   train {lgb_bow_tr:.1f}s | infer {lgb_bow_inf:.2f}s | F1 {f1_score(y_val.astype(int), lgb_bow_preds, average="macro"):.4f}')

# 3. Twitter-RoBERTa Fine-Tuning

`cardiffnlp/twitter-roberta-base` — pre-trained on 58 M tweets.

In [ ]:
%%time
model_name = 'cardiffnlp/twitter-roberta-base'
tokenizer  = AutoTokenizer.from_pretrained(model_name)

categories = ['intentionalism', 'anti_intentionalism', 'cognitivism', 'expressivism', 'hedonism', 'originality', 'achievement', 'none']
label2id = {cat: i for i, cat in enumerate(categories)}
id2label = {i: cat for i, cat in enumerate(categories)}

def tokenize(batch):
    return tokenizer(batch['text'], padding='max_length', truncation=True, max_length=128)

hf_train = Dataset.from_dict({
    'text':  X_tr.tolist(),
    'label': y_tr.astype(float).tolist()}).map(tokenize, batched=True)
hf_val = Dataset.from_dict({
    'text':  X_val.tolist(),
    'label': y_val.astype(float).tolist()}).map(tokenize, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=len(categories), id2label=id2label, label2id=label2id, problem_type="multi_label_classification")

args = TrainingArguments(
    output_dir='./roberta_results',
    evaluation_strategy='epoch', save_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=16, per_device_eval_batch_size=32,
    num_train_epochs=3, weight_decay=0.01, load_best_model_at_end=True)

def compute_metrics(ep):
    logits, labels = ep
    probs = 1 / (1 + np.exp(-logits))  # sigmoid
    preds = (probs > 0.5).astype(int)
    return {'f1_macro': f1_score(labels, preds, average='macro'), 'subset_accuracy': accuracy_score(labels, preds)}

trainer = Trainer(model=model, args=args,
                  train_dataset=hf_train, eval_dataset=hf_val,
                  compute_metrics=compute_metrics)

In [ ]:
%%time
t0 = time.time()
trainer.train()
print(f'RoBERTa train time: {time.time()-t0:.1f}s')

res = trainer.evaluate()
print(f'RoBERTa val accuracy: {res["eval_accuracy"]:.4f}')

best_path = classifiers_folder / 'best_roberta_model'
trainer.save_model(str(best_path))
tokenizer.save_pretrained(str(best_path))
print(f'Model saved to {best_path}')

# 4. Predict Next Batch and Export Active-Learning Sample

In [ ]:
%%time
assert NEXT_BATCH_PATH.exists(), f'Batch not found: {NEXT_BATCH_PATH}'

pending = pd.read_pickle(NEXT_BATCH_PATH)
print(f'Running inference on {len(pending):,} tweets...')

clf_pipe = pipeline('text-classification', model=trainer.model,
                    tokenizer=tokenizer,
                    device=0 if torch.cuda.is_available() else -1,
                    return_all_scores=True)

preds = []
t0 = time.time()
for i in range(0, len(pending), 500):
    preds.extend(clf_pipe(pending['text'].iloc[i:i+500].astype(str).tolist()))
print(f'Inference done in {time.time()-t0:.1f}s')

categories = ['intentionalism', 'anti_intentionalism', 'cognitivism', 'expressivism', 'hedonism', 'originality', 'achievement', 'none']
for cat in categories:
    pending[f'pred_{cat}'] = [[1 if y['score'] > 0.5 else 0 for y in s if y['label'] == cat][0] for s in preds]
    
# Confidence could be the average confidence of the positive predictions, or just the max score
pending['confidence'] = [max(x['score'] for x in s) for s in preds]

In [ ]:
%%time
N_UNCERTAIN = 5_000
N_RANDOM    = 5_000

uncertain = pending.nsmallest(min(N_UNCERTAIN, len(pending)), 'confidence')
pool      = pending.drop(uncertain.index)
random_s  = pool.sample(n=min(N_RANDOM, len(pool)), random_state=42)

export = pd.concat([uncertain, random_s]).sample(frac=1, random_state=42)
export['human_intentionalism'] = np.nan
export['human_anti_intentionalism'] = np.nan
export['human_cognitivism'] = np.nan
export['human_expressivism'] = np.nan
export['human_hedonism'] = np.nan
export['human_originality'] = np.nan
export['human_achievement'] = np.nan
export['human_none'] = np.nan
export['text'] = export['text'].astype(str).str.replace('\n', ' ', regex=False)

out = hitl_folder / f'hitl_review_batch_{PENDING_BATCH_TO_PROCESS:02d}.csv'
export.to_csv(out, index=False)
print(f'Exported {len(export):,} tweets for review to {out}')
print('Next: fill human_label, save, increment PENDING_BATCH_TO_PROCESS, re-run.')

In [ ]:
# Disconnect from Colab runtime (no-op locally)
try:
    from google.colab import runtime
    runtime.unassign()
except ImportError:
    pass
